# Event-type composition (lumi-weighted)

ZH, HH, Single Higgs, and other contributions in each dataset, normalised with `Lumi_weight`. Outputs: `plots/event_type_contribution_*.png`.


In [ ]:
import os
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import numpy as np

os.makedirs('plots', exist_ok=True)

def load_dataset(filepath):
    with h5py.File(filepath, 'r') as f:
        return pd.DataFrame.from_records(f['ForAnalysis/1d'][:])

sm_df = load_dataset('datasets/new_Input_bbyy_SMEFT_SM_4thMarch_2026.h5')
cbbim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbbim_4thMarch_2026.h5')
cbgim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbgim_4thMarch_2026.h5')
cbhim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbhim_4thMarch_2026.h5')
chbtil_df = load_dataset('datasets/new_Input_bbyy_SMEFT_chbtil_4thMarch_2026.h5')
chgtil_df = load_dataset('datasets/new_Input_bbyy_SMEFT_chgtil_4thMarch_2026.h5')
ctbim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_ctbim_4thMarch_2026.h5')

datasets = {
    'SM': sm_df, 'cbbim': cbbim_df, 'cbgim': cbgim_df,
    'cbhim': cbhim_df, 'chbtil': chbtil_df, 'chgtil': chgtil_df, 'ctbim': ctbim_df
}
colors = {'SM': 'blue', 'cbbim': 'orange', 'cbgim': 'red', 'cbhim': 'green',
          'chbtil': 'purple', 'chgtil': 'brown', 'ctbim': 'magenta'}

### Lumi-weighted contribution by event type

In [ ]:
event_types = {
    'ZH': 'is_ZHEvent',
    'HH': 'is_HHEvent',
    'Single H': 'is_SingleHiggsEvent',
}

results = []
for name, df in datasets.items():
    w = df['Lumi_weight'].values
    total_weight = w.sum()
    row = {'Dataset': name, 'Total weight': total_weight, 'N events': len(df)}
    for label, col in event_types.items():
        mask = df[col] == True
        w_sum = w[mask].sum()
        pct = 100 * w_sum / total_weight if total_weight > 0 else 0
        row[label] = pct
        row[f'{label} (weight)'] = w_sum
    # Other = events not in ZH, HH, or Single H
    other_mask = ~(df['is_ZHEvent'] | df['is_HHEvent'] | df['is_SingleHiggsEvent'])
    row['Other'] = 100 * w[other_mask].sum() / total_weight if total_weight > 0 else 0
    results.append(row)

results_df = pd.DataFrame(results)
print(f"\n{'='*70}")
print("Lumi-weighted contribution (% by event type)")
print(f"{'='*70}")
display_cols = ['Dataset', 'N events', 'ZH', 'HH', 'Single H', 'Other']
print(results_df[display_cols].to_string(index=False))
results_df

### Stacked bar chart: ZH, HH, Single H, Other (%)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(datasets))
width = 0.6

bottom = np.zeros(len(datasets))
bars = []
bar_colors = ['#2ecc71', '#f39c12', '#3498db', '#95a5a6']
for i, (label, col) in enumerate([('ZH', 'ZH'), ('HH', 'HH'), ('Single H', 'Single H'), ('Other', 'Other')]):
    vals = results_df[label].values
    p = ax.bar(x, vals, width, bottom=bottom, label=label, color=bar_colors[i])
    bars.append(p)
    bottom += vals

ax.set_ylabel('Lumi-weighted contribution (%)', fontsize=12)
ax.set_xlabel('Dataset', fontsize=12)
ax.set_title('Event type contribution by dataset (Lumi_weight)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(results_df['Dataset'])
ax.legend(loc='upper right', fontsize=10)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('plots/event_type_contribution_stacked.png', dpi=150, bbox_inches='tight')
plt.show()

### Grouped bar chart (side-by-side comparison)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(datasets))
width = 0.2

for i, (label, offset) in enumerate([('ZH', -1.5), ('HH', -0.5), ('Single H', 0.5), ('Other', 1.5)]):
    vals = results_df[label].values
    ax.bar(x + offset * width, vals, width, label=label)

ax.set_ylabel('Lumi-weighted contribution (%)', fontsize=12)
ax.set_xlabel('Dataset', fontsize=12)
ax.set_title('Event type contribution by dataset (Lumi_weight)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(results_df['Dataset'])
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('plots/event_type_contribution_grouped.png', dpi=150, bbox_inches='tight')
plt.show()

### Summary table (raw counts vs weighted)

In [ ]:
print("\nRaw counts vs Lumi-weighted percentages:")
print("\nRaw counts (N events):")
for name, df in datasets.items():
    zh = df['is_ZHEvent'].sum()
    hh = df['is_HHEvent'].sum()
    sh = df['is_SingleHiggsEvent'].sum()
    other = len(df) - zh - hh - sh
    print(f"  {name}: ZH={zh:,}, HH={hh:,}, Single H={sh:,}, Other={other:,}")

print("\nLumi-weighted %:")
for _, row in results_df.iterrows():
    print(f"  {row['Dataset']}: ZH={row['ZH']:.1f}%, HH={row['HH']:.1f}%, Single H={row['Single H']:.1f}%, Other={row['Other']:.1f}%")